In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


# Data Exploration with Pandas
In this notebook we ll try to find insight from data on births in the United States, provided by the Centers for Disease Control (CDC). We analyze data with various methods of ***python***'s most popular library ***pandas***. To visualize the analysis we sometime apply another two popular library *matplotlib* and *seaborn*. Let's get started.

In [3]:
# %load_ext cudf.pandas#Importing all necessary libraries

import numpy as np
import pandas as pd
import seaborn as sns# %matplotlib inline
import matplotlib.pyplot as plt
import time

In [4]:
start_time = time.time()

In [5]:
### cell 0 ###
#Alternatively you can use this code to get the dataset# url = "https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv"
birth_data = pd.read_csv("/home/dias-benchmarks/new_notebooks/us-birth/births.csv")
birth_data.sample(5, random_state=0)

,year,month,day,gender,births
11173,1983,9,24,F,4477
6263,1977,3,12,F,4053
6381,1977,5,8,M,3967
4392,1974,9,28,M,4731
4782,1975,4,1,F,4263


If you are seeing df.sample() first time, I tell you df.sample(5) as opposed to df.head(5) selects five random rows, thus giving you a more unbiased view of the data. The sample() method returns 1 row if a number is not specified. The column names will also be returned, in addition to the sample rows. In that case we have to add axis = 1 like df.sample(axis = 1) returns any random column.

In [6]:
### cell 1 ###

factor = 500
birth_data = pd.concat([birth_data] * factor)

# Data Exploration

In [7]:
### cell 2 ###

birth_data.shape, birth_data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7773500 entries, 0 to 15546
Data columns (total 5 columns):
 #   Column  Dtype
---  ------  -----
 0   year    int64
 1   month   int64
 2   day     int64
 3   gender  object
 4   births  int64
dtypes: int64(4), object(1)
memory usage: 334.5+ MB


((7773500, 5), None)

We see the day column is representing days for every month. It's better to come with integer data than float. So we change here it's data type into int64. See the following results. 

In [8]:
### cell 3 ###

birth_data = birth_data.fillna(0)
birth_data['day'] = birth_data['day'].astype(int)
birth_data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7773500 entries, 0 to 15546
Data columns (total 5 columns):
 #   Column  Dtype
---  ------  -----
 0   year    int64
 1   month   int64
 2   day     int64
 3   gender  object
 4   births  int64
dtypes: int64(4), object(1)
memory usage: 333.6+ MB


In [9]:
### cell 4 ###

birth_data.sample(5, random_state=0)

,year,month,day,gender,births
7612,1978,12,19,M,5436
721,1969,12,9,M,5484
6585,1977,8,14,M,4170
13249,1986,7,14,F,5364
12572,1985,8,11,M,4496


**Adding an extra column:** As the years are repeatedly there in 'year' column, to make the data analysis easy and insight of the data look natural let's add a decade column. By this all the records with year = [1960, 1961, ... , 1969] goes to decade - 1960, simillary, all rows with year = [1970, 1971, ... , 1979] goes to decade 1970 and so on. Later we see it's advantage.

In [10]:
### cell 5 ###
#Code to add the decade column 
birth_data['decade'] = 10 * (birth_data['year'] // 10)
birth_data.head()

,year,month,day,gender,births,decade
0,1969,1,1,F,4046,1960
1,1969,1,1,M,4440,1960
2,1969,1,2,F,4454,1960
3,1969,1,2,M,4548,1960
4,1969,1,3,F,4548,1960


In [11]:
### cell 6 ###
#take a look at male and female births as a function of decade
birth_data.pivot_table('births', index='decade', columns='gender', aggfunc='sum')

gender,F,M
decade,,
1960,876817000,923286000
1970,8131537500,8560775000
1980,9155175500,9621726000
1990,9739727000,10210276500
2000,9114654500,9553214000


We immediately see that male births outnumber female births in every decade. To see this trend a bit more clearly, we can use the built-in plotting tools in Pandas to visualize the total number of births by year.

In [12]:
# %matplotlib inline# import matplotlib.pyplot as plt# plt.rcParams["figure.figsize"] = (10,6)# sns.set()  # use Seaborn styles# birth_data.pivot_table('births', index='year', columns='gender', aggfunc='sum').plot()# plt.ylabel('total births per year')# plt.title("Distribution Of Births Over Decades")

With a simple pivot table and built-in plot() method, we can immediately see the annual trend in births by gender. By eye, it appears that over the past 40 years male births have outnumbered female births by around 5%.

At this point anyone can make conclusion about the final trend of births. But this figure is not telling everything. Let us go into deep to explore further. Look at the code/output of two cells below.

In [13]:
# %matplotlib inline# #import matplotlib.pyplot as plt# plt.rcParams["figure.figsize"] = (10,6)# sns.set()  # use Seaborn styles# birth_data.pivot_table('births', index='year', columns='gender', aggfunc='mean').plot()# plt.ylabel('Mean births per year')# plt.title("Distribution Of Mean Births Over Decades")

What do you say now. There is drastic increase in mean birth rate when we move from eighties towards ninties. This was unimaginable from the previous figure where aggregator function was sum(). Was it realy happened? Look at the numerical table

In [14]:
### cell 7 ###
#Code to present male and female mean(births) against decade numericaly
birth_data.pivot_table('births', index='decade', columns='gender', aggfunc='mean')

gender,F,M
decade,,
1960,4566.755208,4808.781250
1970,4267.403569,4497.386393
1980,5460.886072,5740.886635
1990,162328.783333,170171.275000
2000,168789.898148,176911.370370


Look at the last two rows in the pivot table above. Values are arround 30 times higher than previous values. Was it real or there is some irregularities in the dataset? We see few more charts/tables.



In [15]:
# #Boxplot for births# #% matplotlib inline# #import matplotlib.pyplot as plt# sns.boxplot(y = 'births', data = birth_data)# plt.title('Box plot - Total births')

No sufficient information gain from this figure, it seems a lot of outliers are there and maximum of them towards upper side that is year with high total births. Let's customize this box plot over decade. It may help us.

In [16]:
# #boxplot of births by decade# fig, ax = plt.subplots(figsize=(10, 8))# sns.boxplot(x = 'decade', y = 'births', data = birth_data, ax = ax)# plt.title('Box plot - Total births over decade')# plt.show()

Now it's clear they are not all outliers. Actualy change of distribution is happening when we enter nineties.

In [17]:
### cell 8 ###

years2check = [1988, 1989]

for year in years2check:
  yearly_data = birth_data[birth_data['year'] == year]
  print("Births data for - ", year)
  print(yearly_data)

Births data for -  1988
       year  month  day gender  births  decade
14328  1988      1    1      F    4149    1980
14329  1988      1    1      M    4345    1980
14330  1988      1    2      F    3874    1980
14331  1988      1    2      M    4175    1980
14332  1988      1    3      F    3981    1980
...     ...    ...  ...    ...     ...     ...
15062  1988     12   29      M    5944    1980
15063  1988     12   30      F    5742    1980
15064  1988     12   30      M    6095    1980
15065  1988     12   31      F    4435    1980
15066  1988     12   31      M    4698    1980

[369500 rows x 6 columns]
Births data for -  1989
       year  month  day gender  births  decade
15067  1989      1    0      F  156749    1980
15068  1989      1    0      M  164052    1980
15069  1989      2    0      F  146710    1980
15070  1989      2    0      M  154047    1980
15071  1989      3    0      F  165889    1980
...     ...    ...  ...    ...     ...     ...
15086  1989     10    0      M  

**Note:** Two major changes in data collection we observeed. First: from year 1969 to 1988 data collected/stored day wise for male and female separately. But from year 1989 onwards data collected monthwise for male and female separately i.e. for every year in [1989, 1990, ... , 2008] there are 24 records are there of which 12 for male and remaining 12 for female. Second: the day column has value day wise i.e. from [1, 2, ... , 31] till year 1988  with extra two column with value 99 for each male and female while from year 1989 onwards day column has value zero (initial NAN value, we changed to zero). That is why there are 480 = 24 * 20 zero values there, from 1989 to 2008 is 20 years.

Change in data collection started from 1989. Now if you look again the above box plot we understand that it'd be better we analyze data separately. Let's get into the code.

In [18]:
### cell 9 ###
#Descriptive summary before removing outliers
birth_data.describe()

,year,month,day,births,decade
count,7.773500e+06,7.773500e+06,7.773500e+06,7.773500e+06,7.773500e+06
mean,1.979037e+03,6.515919e+00,1.722126e+01,9.762294e+03,1.974544e+03
std,6.728125e+00,3.449521e+00,1.535651e+01,2.855155e+04,6.789365e+00
min,1.969000e+03,1.000000e+00,0.000000e+00,1.000000e+00,1.960000e+03
25%,1.974000e+03,4.000000e+00,8.000000e+00,4.358000e+03,1.970000e+03
50%,1.979000e+03,7.000000e+00,1.600000e+01,4.814000e+03,1.970000e+03
75%,1.984000e+03,1.000000e+01,2.400000e+01,5.290000e+03,1.980000e+03
max,2.008000e+03,1.200000e+01,9.900000e+01,1.996220e+05,2.000000e+03


In [19]:
%%time
### cell 10 ###



# Assuming 'birth_data' is a cuDF DataFrame
# For example:
# data = np.random.normal(loc=4000, scale=500, size=10000)
# birth_data = cudf.DataFrame({'births': data})

# 1. Your original setup code is fine
#    (Use .to_numpy() to transfer data to the CPU for the numpy function)
quartiles = np.percentile(birth_data['births'].to_numpy(), [25, 50, 75])
mu = quartiles[1]
sig = 0.74 * (quartiles[2] - quartiles[0])

# 2. Pre-calculate the final boundary values
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig

# 3. Create the query string using an f-string
#    This injects the *values* of lower_bound and upper_bound into the string
query_string = f"(births > {lower_bound}) & (births < {upper_bound})"

# 4. Use the generated string in the query() method
births = birth_data.query(query_string)

# The rest of your code works as expected
birth_data.shape, births.sample(5, random_state=0)### cell 10 ###

# Compute quartiles on GPU
quartiles = birth_data['births'].quantile([0.25, 0.5, 0.75])
q25, mu, q75 = quartiles.iloc[0], quartiles.iloc[1], quartiles.iloc[2]

# Sigma and bounds (still small Python scalars)
sig = 0.74 * (q75 - q25)
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig

# Filter on GPU via boolean indexing
births = birth_data[(birth_data['births'] > lower_bound) & (birth_data['births'] < upper_bound)]

birth_data.shape, births.sample(5, random_state=0)

((7773500, 6),
        year  month  day gender  births  decade
 5069   1975      8   16      M    4342    1970
 1750   1971      4   13      M    5366    1970
 2793   1972      8   25      F    4797    1970
 13703  1987      2   24      F    5396    1980
 7636   1978     12   31      M    4202    1970)

In [20]:
# #Code to remove outliers or actualy to consider one part of the data # quartiles = np.percentile(birth_data['births'], [25, 50, 75])# mu = quartiles[1]# sig = 0.74 * (quartiles[2] - quartiles[0])# births = birth_data.query('(births > mu - 5 * sig) & (births < mu + 5 * sig)')# print(births.shape)# births.sample(5)

In [21]:
### cell 11 ###
#Descriptive summary after removal of outliers
births.describe()

,year,month,day,births,decade
count,7.305000e+06,7.305000e+06,7.305000e+06,7.305000e+06,7.305000e+06
mean,1.978501e+03,6.522930e+00,1.572964e+01,4.824470e+03,1.974001e+03
std,5.766341e+00,3.448703e+00,8.800093e+00,5.799772e+02,5.830600e+00
min,1.969000e+03,1.000000e+00,1.000000e+00,3.249000e+03,1.960000e+03
25%,1.974000e+03,4.000000e+00,8.000000e+00,4.383000e+03,1.970000e+03
50%,1.979000e+03,7.000000e+00,1.600000e+01,4.812000e+03,1.970000e+03
75%,1.984000e+03,1.000000e+01,2.300000e+01,5.259000e+03,1.980000e+03
max,1.988000e+03,1.200000e+01,3.100000e+01,6.527000e+03,1.980000e+03


If we compare these two summaries we observe the major difference is in births column. Before mean, std = 9762.3, 28552.5; after mean, std = 4824.5, 580. One more thing to say: this new data births(saved after removing outliers) must have years up to 1988. Let's run the following code.

In [22]:
### cell 12 ###

births.year.unique()

array([1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979,
       1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988])

Finally, we can combine the day, month, and year to create a Date index. This allows us to quickly compute the weekday corresponding to each row. Think about the records after 1988 where day val is zero for all rows. So we could not make this action.

In [23]:
### cell 13 ###
# create a datetime index from the year, month, day
births.index = pd.to_datetime(10000 * births.year + 100 * births.month + births.day, format='%Y%m%d')
births.head()

,year,month,day,gender,births,decade
1969-01-01,1969,1,1,F,4046,1960
1969-01-01,1969,1,1,M,4440,1960
1969-01-02,1969,1,2,F,4454,1960
1969-01-02,1969,1,2,M,4548,1960
1969-01-03,1969,1,3,F,4548,1960


In [24]:
### cell 14 ###
#Creating another column 'dayofweek' using dayofweek attribute of pandas DatetimeIndex
births['dayofweek'] = births.index.dayofweek
births.head()

,year,month,day,gender,births,decade,dayofweek
1969-01-01,1969,1,1,F,4046,1960,2
1969-01-01,1969,1,1,M,4440,1960,2
1969-01-02,1969,1,2,F,4454,1960,3
1969-01-02,1969,1,2,M,4548,1960,3
1969-01-03,1969,1,3,F,4548,1960,4


In [25]:
### cell 15 ###

mapping = {0: 'Monday', 1: 'Tuesady', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
births['dayofweek'] = births['dayofweek'].map(mapping)

In [26]:
### cell 16 ###

births.head()

,year,month,day,gender,births,decade,dayofweek
1969-01-01,1969,1,1,F,4046,1960,Wednesday
1969-01-01,1969,1,1,M,4440,1960,Wednesday
1969-01-02,1969,1,2,F,4454,1960,Thursday
1969-01-02,1969,1,2,M,4548,1960,Thursday
1969-01-03,1969,1,3,F,4548,1960,Friday


In [27]:
# # plot for births by week days over decades# %matplotlib inline# import matplotlib.pyplot as plt# plt.rcParams["figure.figsize"] = (10,6)# sns.set()  # use Seaborn styles# births.pivot_table('births', index='dayofweek', columns='decade', aggfunc='mean').plot()# plt.ylabel('mean births by day')# plt.title("Trend Of Mean Births Along Week Days Over Decades ")

What do we conclude here? Births are slightly less common on weekends than on weekdays! It is common since decades. Note that unless we removed 1990s and 2000s decades we could not have this plot because the CDC data contains only the month of birth starting from 1989.

Our last attempt is to view the plot as the mean number of births by the day of the year over decades. Let's first group the data by month and day separately. so that dataframe will have 366 (+1 for leap year) records where first value represents the mean births of all births on 1st Jan from 1969 to 1988.

In [28]:
### cell 17 ###

births_by_date = births.pivot_table('births', [births.index.month, births.index.day])
births_by_date.sample(10, random_state=0)

,,births
6,15,4750.475
1,10,4591.700
10,26,4788.050
9,25,5240.000
3,7,4722.450
10,17,4818.850
11,30,4765.475
10,2,5103.625
2,15,4706.325
10,18,4751.625


The first row above interpretes that mean births on 1st October from 1969 to 1988 is 5167.325 and the same way other rows can be interpreted. What we created above is a multi index dataframe which have total 366 rows.

In [29]:
### cell 18 ###

births_by_date.shape

(366, 1)

To plot this data easily we turn these month-day multi index into a date (like time series) by associating them with a dummy year variable (say) 2020. In fact you can take any leap year.

In [30]:
### cell 19 ###
#Creating index like timeseries# births_by_date.index = [pd.datetime(2020, month, day) for (month, day) in births_by_date.index]# births_by_date.head()
births_by_date.index = [
    pd.Timestamp(2020, month, day)
    for (month, day) in births_by_date.index
]
births_by_date.head()

,births
2020-01-01,4009.225
2020-01-02,4247.400
2020-01-03,4500.900
2020-01-04,4571.350
2020-01-05,4603.625


In [31]:
end_time = time.time()
print(end_time - start_time)

11.81145191192627


So we have a dataframe (like time series) with only values the average number of births by date of the year. Now we can use the simple DataFrame.plot() method to plot the data. let's see the code/output.

In [32]:
# # Plot the results# fig, ax = plt.subplots(figsize=(12, 6))# births_by_date.plot(ax=ax)